# Kapitola 4: Oddělování dat a instrukcí

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Experimentální playground](#example-playground)

## Nastavení

Spusťte následující nastavovací buňku, která načte API klíč a vytvoří pomocnou funkci `get_completion`.


In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Lekce

Často nechceme psát celé prompty ručně, ale chceme **šablony promptů, které lze později upravit doplněním vstupních dat před odesláním Claude**. To se hodí, když chcete, aby Claude pokaždé dělal stejnou věc, ale data, se kterými pracuje, se pokaždé liší.

Naštěstí to jde poměrně snadno: **oddělíme pevnou kostru promptu od proměnlivého uživatelského vstupu a potom uživatelský vstup dosadíme do promptu** před odesláním celého promptu Claude.

Níže si krok za krokem ukážeme, jak napsat dosaditelnou šablonu promptu a jak do ní dosadit uživatelský vstup.


### Příklady

V prvním příkladu žádáme Claude, aby fungoval jako generátor zvuků zvířat. Všimněte si, že celý prompt odeslaný Claude je jen `PROMPT_TEMPLATE` s dosazeným vstupem, v tomto případě `Cow`. Slovo `Cow` nahradí placeholder `ANIMAL` pomocí f-stringu, když vypisujeme celý prompt.

**Poznámka:** V praxi nemusíte placeholder proměnnou pojmenovat nijak konkrétně. V tomto příkladu jsme ji nazvali `ANIMAL`, ale stejně dobře bychom ji mohli nazvat `CREATURE` nebo `A`. Obecně je ale lepší používat konkrétní a relevantní názvy proměnných, aby byla šablona promptu srozumitelná i bez dosazení. Jen se ujistěte, že název proměnné v šabloně odpovídá tomu, co používáte ve f-stringu.


In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Proč bychom chtěli vstupy takto oddělovat a dosazovat? Protože **šablony promptů zjednodušují opakující se úlohy**. Představte si, že postavíte strukturu promptu, do které mohou uživatelé třetích stran zadávat obsah, například zvíře, jehož zvuk chtějí vygenerovat. Tito uživatelé nemusí psát ani vidět celý prompt. Stačí jim vyplnit proměnné.

Zde dosazování děláme pomocí proměnných a f-stringů, ale lze použít i metodu `format()`.

**Poznámka:** Šablony promptů mohou mít libovolný počet proměnných.


Když zavádíte dosazovací proměnné, je velmi důležité **zajistit, aby Claude věděl, kde proměnné začínají a končí** oproti instrukcím nebo popisu úkolu. Podívejme se na příklad, kde instrukce a dosazovaná proměnná nejsou oddělené.

Lidským očím je v šabloně promptu níže jasné, kde proměnná začíná a končí. Po úplném dosazení promptu se ale toto ohraničení stane nejasným.


In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Zde si **Claude myslí, že `Yo Claude` je součást e-mailu, který má přepsat**. Poznáte to podle toho, že přepis začne slovy `Dear Claude`. Pro člověka je jasné, hlavně v šabloně promptu, kde e-mail začíná a končí. Po dosazení do promptu je to ale mnohem méně jasné.


Jak to vyřešíme? **Obalte vstup XML tagy.** Udělali jsme to níže a jak vidíte, ve výstupu už není `Dear Claude`.

[XML tagy](https://docs.anthropic.com/claude/docs/use-xml-tags) jsou tagy v lomených závorkách, například `<tag></tag>`. Vyskytují se v párech a skládají se z otevíracího tagu, například `<tag>`, a zavíracího tagu označeného `/`, například `</tag>`. XML tagy se používají k obalení obsahu, například takto: `<tag>content</tag>`.

**Poznámka:** Claude dokáže rozpoznat a používat širokou škálu oddělovačů, ale doporučujeme **používat konkrétně XML tagy jako oddělovače** pro Claude, protože Claude byl specificky trénován rozpoznávat XML tagy jako mechanismus pro organizaci promptu. Mimo function calling **neexistují žádné speciální XML tagy, na které by Claude byl natrénovaný a které by magicky maximalizovaly výkon**. Claude je záměrně velmi tvárný a přizpůsobitelný.


In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Podívejme se na další příklad, kde XML tagy pomohou.

V následujícím promptu **Claude nesprávně interpretuje, která část promptu je instrukce a která je vstup**. Kvůli formátování nesprávně považuje `Each is about an animal, like rabbits` za součást seznamu, i když uživatel, který vyplňuje proměnnou `SENTENCES`, to pravděpodobně nechtěl.


In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Abychom to opravili, stačí **obklopit uživatelské vstupní věty XML tagy**. Tím Claude ukážeme, kde vstupní data začínají a končí, navzdory zavádějící pomlčce před `Each is about an animal, like rabbits.`


In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**Poznámka:** V nesprávné verzi promptu `Each is about an animal` jsme museli zahrnout pomlčku, aby Claude odpověděl nesprávně způsobem, který jsme pro příklad chtěli. To je důležitá lekce o promptování: **na drobných detailech záleží**. Vždy stojí za to **projít prompty a odstranit překlepy a gramatické chyby**. Claude je citlivý na vzory; v raných letech před finetuningem šlo o nástroj pro predikci textu. Je pravděpodobnější, že bude dělat chyby, když je děláte vy, bude chytřejší, když budete znít chytřeji, bláznivější, když budete znít bláznivěji, a podobně.

Pokud chcete experimentovat s prompty z lekce, aniž byste měnili obsah výše, sjeďte až na konec notebooku do části [**Example Playground**](#example-playground).


---

## Cvičení
- [Cvičení 4.1 - Téma haiku](#exercise-41---haiku-topic)
- [Cvičení 4.2 - Otázka o psech s překlepy](#exercise-42---dog-question-with-typos)
- [Cvičení 4.3 - Otázka o psech, část 2](#exercise-42---dog-question-part-2)


### Cvičení 4.1 - Téma haiku
Upravte `PROMPT` tak, aby šlo o šablonu, která přijme proměnnou `TOPIC` a vygeneruje haiku o daném tématu. Toto cvičení slouží jen k ověření, že rozumíte struktuře šablon s proměnnými ve f-stringu.


In [ ]:
# Variable content
TOPIC = "Pigs"

# Prompt template with a placeholder for the variable content
PROMPT = f""

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_4_1_hint; print(exercise_4_1_hint)

### Cvičení 4.2 - Otázka o psech s překlepy
Opravte `PROMPT` přidáním XML tagů tak, aby Claude vytvořil správnou odpověď.

Snažte se neměnit nic dalšího v promptu. Neupravené a chybami zatížené psaní je záměrné, abyste viděli, jak Claude na takové chyby reaguje.


In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_4_2_hint; print(exercise_4_2_hint)

### Cvičení 4.3 - Otázka o psech, část 2
Opravte `PROMPT` **BEZ** přidání XML tagů. Místo toho odstraňte pouze jedno nebo dvě slova z promptu.

Stejně jako u předchozích cvičení se snažte neměnit nic dalšího. Ukáže vám to, jaký typ jazyka Claude dokáže parsovat a chápat.


In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_4_3_hint; print(exercise_4_3_hint)

### Gratulujeme!

Pokud jste vyřešili všechna cvičení až sem, jste připraveni přejít na další kapitolu. Hodně zdaru s promptováním.


---

## Example Playground

Toto je prostor, kde můžete volně experimentovat s příklady promptů z této lekce a upravovat je, abyste viděli, jak se tím mění odpovědi Claude.


In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))